# 🧪 TabuLab — Synthetic Tabular Data Generator using CTGAN

**College Project PS 65** | SDV 2.0 | CTGAN | Python

---

## Notebook Overview

This notebook walks through the complete pipeline:

| Stage | Description |
|---|---|
| A | Import libraries & load dataset |
| B | Exploratory Data Analysis (EDA) |
| C | Preprocessing & SDV Metadata |
| D | Train CTGAN synthesizer |
| E | Generate & export synthetic data |
| F | Real vs Synthetic comparison |
| G | ML utility evaluation (TSTR) |
| H | Privacy risk checks |
| I | Final summary & limitations |

> **Run cells top-to-bottom.** Each section has a Markdown explanation followed by executable code.

---
## Stage A — Import Libraries & Load Dataset

In [ ]:
!pip install sdv==2.0.0 scikit-learn seaborn

In [ ]:
# ── Standard library & third-party imports ────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    mean_absolute_error, mean_squared_error, classification_report
)
from sklearn.preprocessing import LabelEncoder

# ── SDV 2.0 imports ───────────────────────────────────────────────────────────
import sdv
from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer

# ── sdmetrics for quality report ──────────────────────────────────────────────
from sdmetrics.reports.single_table import QualityReport

print(f'SDV version  : {sdv.__version__}')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version : {np.__version__}')
print('All imports OK ✅')

### A1 — Choose Dataset

Two options:
- **Option 1 (default):** Load the public **Titanic** dataset from seaborn.
- **Option 2:** Upload your own CSV.

Set `USE_OWN_CSV = True` and `CSV_PATH` to use your own file.

In [ ]:
# ─── CONFIGURATION ────────────────────────────────────────────────────────────
USE_OWN_CSV = False          # Set True to use your own CSV
CSV_PATH    = 'data/my_data.csv'   # Path to your CSV (if USE_OWN_CSV = True)
TABLE_NAME  = 'table'        # Internal table name (do not change)
RANDOM_SEED = 42

# CTGAN Training
EPOCHS      = 100            # Recommended: 100–300 for good quality
BATCH_SIZE  = 200

# Synthetic generation
N_SYNTHETIC = 500            # Number of synthetic rows to generate

# Train/test split
TEST_SIZE   = 0.20           # 20% held out as real test set

# ML evaluation
TARGET_COL  = 'survived'     # Column to predict (change if using your own data)

# Output
OUTPUT_PATH = 'outputs/synthetic_data.csv'
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import os
os.makedirs('outputs', exist_ok=True)
os.makedirs('data', exist_ok=True)

if USE_OWN_CSV:
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f'CSV not found at {CSV_PATH}. Check your path.')
    df_raw = pd.read_csv(CSV_PATH)
    print(f'Loaded your CSV: {CSV_PATH}')
else:
    # Load Titanic from seaborn (public dataset, no API key needed)
    df_raw = sns.load_dataset('titanic')
    # Drop redundant columns that are derived from others
    df_raw = df_raw.drop(
        columns=['alive', 'embark_town', 'who', 'adult_male', 'deck'],
        errors='ignore'
    )
    print('Loaded demo Titanic dataset.')

print(f'Shape: {df_raw.shape}')
df_raw.head()

---
## Stage B — Exploratory Data Analysis (EDA)

We inspect the data before training to understand:
- Data dimensions and column types
- Missing values and duplicates
- Numerical distributions
- Categorical frequency distributions
- Correlations between numerical features

In [ ]:
print('=' * 55)
print(f'  Shape        : {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
print(f'  Duplicate rows: {df_raw.duplicated().sum()}')
print(f'  Total NaN    : {df_raw.isna().sum().sum()}')
print('=' * 55)
df_raw.info()

In [ ]:
# Column-level overview
overview = pd.DataFrame({
    'dtype'       : df_raw.dtypes,
    'non_null'    : df_raw.notnull().sum(),
    'missing_%'   : (df_raw.isnull().mean() * 100).round(2),
    'unique'      : df_raw.nunique(),
})
print(overview.to_string())

In [ ]:
# Identify numerical and categorical columns
num_cols_raw = df_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_raw = df_raw.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
print(f'Numerical columns ({len(num_cols_raw)}): {num_cols_raw}')
print(f'Categorical columns ({len(cat_cols_raw)}): {cat_cols_raw}')

In [ ]:
# Numerical summary statistics
if num_cols_raw:
    display(df_raw[num_cols_raw].describe().T.round(2))

In [ ]:
# Categorical value counts
for c in cat_cols_raw[:6]:  # Show first 6
    print(f'\n── {c} ──')
    print(df_raw[c].value_counts().head(10).to_string())

In [ ]:
# Numerical histograms
n = min(len(num_cols_raw), 9)
if n > 0:
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4)) if n > 1 else plt.subplots(1, 1, figsize=(5, 4))
    axes = np.array([axes]).flatten()
    for i, c in enumerate(num_cols_raw[:n]):
        df_raw[c].dropna().hist(ax=axes[i], bins=25, color='steelblue', edgecolor='white')
        axes[i].set_title(c)
    plt.suptitle('Numerical Column Distributions (Raw Data)', y=1.02, fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Missing value heatmap
if df_raw.isnull().any().any():
    plt.figure(figsize=(12, 4))
    sns.heatmap(df_raw.isnull(), yticklabels=False, cbar=False, cmap='magma')
    plt.title('Missing Value Map (yellow = missing)')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values!')

In [ ]:
# Correlation heatmap
if len(num_cols_raw) >= 2:
    plt.figure(figsize=(min(len(num_cols_raw), 10), min(len(num_cols_raw), 8)))
    corr = df_raw[num_cols_raw].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, annot_kws={'size': 9})
    plt.title('Correlation Matrix (Numerical Columns)')
    plt.tight_layout()
    plt.show()

---
## Stage C — Preprocessing & SDV Metadata

### Why preprocess?
CTGAN requires clean data. We:
1. **Drop** columns with > 80% missing values (they provide too little signal).
2. **Fill numerical NaN** with the column median (robust to outliers).
3. **Fill categorical NaN** with the column mode.
4. **Split** real data into train (fit CTGAN) and test (evaluate ML utility).
5. **Create SDV Metadata** — SDV needs to know each column's semantic type (numerical, categorical, etc.).

In [ ]:
def preprocess_dataframe(df: pd.DataFrame):
    """Clean DataFrame for CTGAN training."""
    df = df.copy()

    # 1. Drop columns with >80% missing
    thresh = int(len(df) * 0.2)
    df = df.dropna(axis=1, thresh=thresh)

    # 2. Identify column types
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

    # 3. Fill numerical NaN with median
    for c in num_cols:
        df[c] = df[c].fillna(df[c].median())

    # 4. Fill categorical NaN with mode, convert to string
    for c in cat_cols:
        mode_val = df[c].mode()[0] if not df[c].mode().empty else 'Unknown'
        df[c] = df[c].fillna(mode_val).astype(str)

    return df, num_cols, cat_cols


df_clean, num_cols, cat_cols = preprocess_dataframe(df_raw)

print(f'Raw shape   : {df_raw.shape}')
print(f'Clean shape : {df_clean.shape}')
print(f'Numerical   : {num_cols}')
print(f'Categorical : {cat_cols}')
print(f'Remaining NaN: {df_clean.isna().sum().sum()}')

In [ ]:
# ── Train / Test split ────────────────────────────────────────────────────────
# IMPORTANT: We split BEFORE fitting CTGAN.
# CTGAN only sees df_train — df_test is kept strictly for ML evaluation.

df_train, df_test = train_test_split(
    df_clean, test_size=TEST_SIZE, random_state=RANDOM_SEED, shuffle=True
)
print(f'Train rows : {len(df_train):,}  ({(1-TEST_SIZE)*100:.0f}%)')
print(f'Test rows  : {len(df_test):,}   ({TEST_SIZE*100:.0f}%)')

In [ ]:
# ── SDV Metadata ──────────────────────────────────────────────────────────────
# Metadata.detect_from_dataframes() is a CLASSMETHOD that returns a new Metadata object.
# It automatically infers column sdtypes (numerical, categorical, datetime, id, etc.)

metadata = Metadata.detect_from_dataframes({TABLE_NAME: df_train})

print('Tables detected:', list(metadata.tables.keys()))
print('\nMetadata dictionary:')
import json
print(json.dumps(metadata.to_dict(), indent=2, default=str))

---
## Stage D — Train CTGAN Synthesizer

### How CTGAN works (simplified)

CTGAN is a **Generative Adversarial Network** adapted for tabular data.

- **Generator**: Learns to create fake rows that look real.
- **Discriminator**: Tries to tell real rows from fake rows.
- **Conditioning**: The generator is conditioned on sampled categorical values, improving minority-category coverage.

Key parameters:
| Parameter | Meaning | Default |
|---|---|---|
| `epochs` | Training rounds | 300 |
| `batch_size` | Rows per gradient step | 500 |
| `embedding_dim` | Latent space size | 128 |
| `generator_dim` | Generator hidden layers | (256, 256) |
| `discriminator_dim` | Discriminator hidden layers | (256, 256) |

In [ ]:
print(f'Training CTGAN — epochs={EPOCHS}, batch_size={BATCH_SIZE}')
print(f'Training on {len(df_train):,} rows × {df_train.shape[1]} columns')
print('This may take a few minutes…\n')

synthesizer = CTGANSynthesizer(
    metadata,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=True,   # Shows per-epoch loss
)

synthesizer.fit({TABLE_NAME: df_train})

print('\n✅ CTGAN training complete!')

---
## Stage E — Generate Synthetic Data & Export

We call `synthesizer.sample(num_rows=N_SYNTHETIC)` to generate synthetic rows.
The output is a **dict** keyed by table name; we extract the DataFrame.

In [ ]:
print(f'Generating {N_SYNTHETIC:,} synthetic rows…')

syn_dict = synthesizer.sample(num_rows=N_SYNTHETIC)
syn_df   = syn_dict[TABLE_NAME]   # Extract the DataFrame from the dict

print(f'Generated shape: {syn_df.shape}')
syn_df.head(10)

In [ ]:
# Export to CSV
syn_df.to_csv(OUTPUT_PATH, index=False)
print(f'✅ Synthetic data saved to: {OUTPUT_PATH}')

In [ ]:
# Verify the export
loaded_check = pd.read_csv(OUTPUT_PATH)
print(f'Verified CSV load: {loaded_check.shape}')
print(f'Null values in synthetic data: {syn_df.isna().sum().sum()}')

---
## Stage F — Real vs Synthetic Comparison

### F1: SDV Quality Report
Uses `sdmetrics` to compute **Column Shapes** (distribution fidelity) and
**Column Pair Trends** (pairwise correlation fidelity) scores — both on a 0–100% scale.

### F2: Visual Comparisons
Overlay histograms for numerical columns and bar charts for categorical columns.

In [ ]:
# ── SDV Quality Report ────────────────────────────────────────────────────────
# QualityReport.generate() uses the SINGLE_TABLE_V2 metadata dict format
table_meta_dict = metadata.get_table_metadata(TABLE_NAME).to_dict()
single_table_dict = table_meta_dict.get('tables', {}).get(TABLE_NAME, table_meta_dict)

report = QualityReport()
report.generate(df_train, syn_df, single_table_dict, verbose=True)

overall_score = report.get_score()
print(f'\n📊 Overall Quality Score: {overall_score * 100:.1f}%')

In [ ]:
# Per-column details
col_shapes = report.get_details(property_name='Column Shapes')
print('Column Shapes Detail:')
display(col_shapes)

In [ ]:
# ── Numerical histogram overlay ───────────────────────────────────────────────
num_show = [c for c in num_cols if c in syn_df.columns][:9]
if num_show:
    n = len(num_show)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = np.array(axes).flatten()

    for i, c in enumerate(num_show):
        df_train[c].dropna().plot.hist(
            ax=axes[i], bins=25, alpha=0.6, color='steelblue', label='Real'
        )
        syn_df[c].dropna().plot.hist(
            ax=axes[i], bins=25, alpha=0.6, color='tomato', label='Synthetic'
        )
        axes[i].set_title(c)
        axes[i].legend(fontsize=8)

    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Real vs Synthetic — Numerical Distributions', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Categorical proportion bars ───────────────────────────────────────────────
cat_show = [c for c in cat_cols if c in syn_df.columns][:6]
for c in cat_show:
    real_vc = df_train[c].value_counts(normalize=True).head(10)
    syn_vc  = syn_df[c].value_counts(normalize=True).head(10)
    compare = pd.DataFrame({'Real': real_vc, 'Synthetic': syn_vc}).fillna(0)

    ax = compare.plot.bar(figsize=(max(6, len(compare)), 4),
                          color=['steelblue', 'tomato'])
    ax.set_title(f'Category proportions — {c}')
    ax.set_ylabel('Proportion')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Correlation matrix comparison ─────────────────────────────────────────────
common_num = [c for c in num_cols if c in syn_df.columns]
if len(common_num) >= 2:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(df_train[common_num].corr(), ax=ax1, cmap='coolwarm',
                annot=True, fmt='.2f', center=0, annot_kws={'size': 9}, cbar=False)
    ax1.set_title('Real Data Correlations')

    sns.heatmap(syn_df[common_num].corr(), ax=ax2, cmap='coolwarm',
                annot=True, fmt='.2f', center=0, annot_kws={'size': 9}, cbar=False)
    ax2.set_title('Synthetic Data Correlations')

    plt.suptitle('Correlation Matrix: Real vs Synthetic', fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Stage G — ML Utility Evaluation (TSTR)

### Methodology: Train on Synthetic, Test on Real

```
Real data
 ├─ Train split (80%) ──▶ CTGAN trained here
 │                     ├──▶ Synthetic data generated
 │                     │       └──▶ TSTR Model trained on synthetic
 │                     │                └──▶ Evaluated on Real Test Set
 │                     └──▶ Baseline Model trained on real train
 │                               └──▶ Evaluated on Real Test Set
 └─ Test split (20%) ──▶ Evaluation only (never seen by CTGAN)
```

**Target leakage prevention**: `TARGET_COL` is excluded from the feature set during encoding.
The target column in synthetic data comes directly from CTGAN's learned distribution — not from real test labels.

In [ ]:
def encode_for_ml(df: pd.DataFrame, target: str, num_cols: list, cat_cols: list):
    """Label-encode categorical columns and return X, y."""
    feature_cols = [c for c in (num_cols + cat_cols) if c != target]
    X = df[feature_cols].copy()
    for c in cat_cols:
        if c in X.columns:
            X[c] = LabelEncoder().fit_transform(X[c].astype(str))
    y = df[target].copy()
    return X, y


# ── Detect task type ──────────────────────────────────────────────────────────
if TARGET_COL not in df_clean.columns:
    print(f'⚠️  TARGET_COL "{TARGET_COL}" not found. Available: {list(df_clean.columns)}')
    print('Please update TARGET_COL in the configuration cell above.')
else:
    is_classification = (
        df_clean[TARGET_COL].dtype == object
        or df_clean[TARGET_COL].nunique() <= 15
    )
    task_type = 'Classification' if is_classification else 'Regression'
    print(f'Target column : "{TARGET_COL}"')
    print(f'Task type     : {task_type}')
    print(f'Unique values : {df_clean[TARGET_COL].nunique()}')

In [ ]:
if TARGET_COL in df_clean.columns:

    X_real, y_real = encode_for_ml(df_train, TARGET_COL, num_cols, cat_cols)
    X_test, y_test = encode_for_ml(df_test,  TARGET_COL, num_cols, cat_cols)

    # Synthetic may not have the exact same column order — align
    X_syn, y_syn = encode_for_ml(syn_df, TARGET_COL, num_cols, cat_cols)
    X_syn  = X_syn.reindex(columns=X_real.columns, fill_value=0)
    X_test = X_test.reindex(columns=X_real.columns, fill_value=0)

    if is_classification:
        le_target = LabelEncoder()
        y_real = le_target.fit_transform(y_real.astype(str))
        y_syn  = le_target.transform(
            pd.Series(y_syn).astype(str).map(
                lambda x: x if x in le_target.classes_ else le_target.classes_[0]
            )
        )
        y_test = le_target.transform(
            pd.Series(y_test).astype(str).map(
                lambda x: x if x in le_target.classes_ else le_target.classes_[0]
            )
        )

        # ── Baseline: train on real ────────────────────────────────────────────
        baseline = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
        baseline.fit(X_real, y_real)
        y_pred_base = baseline.predict(X_test)

        # ── TSTR: train on synthetic ───────────────────────────────────────────
        tstr_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
        tstr_model.fit(X_syn, y_syn)
        y_pred_tstr = tstr_model.predict(X_test)

        avg = 'binary' if len(le_target.classes_) == 2 else 'macro'

        results = pd.DataFrame({
            'Metric'    : ['Accuracy', 'F1', 'Precision', 'Recall'],
            'Baseline (Real Train)': [
                accuracy_score(y_test, y_pred_base),
                f1_score(y_test, y_pred_base, average=avg, zero_division=0),
                precision_score(y_test, y_pred_base, average=avg, zero_division=0),
                recall_score(y_test, y_pred_base, average=avg, zero_division=0),
            ],
            'TSTR (Synthetic Train)': [
                accuracy_score(y_test, y_pred_tstr),
                f1_score(y_test, y_pred_tstr, average=avg, zero_division=0),
                precision_score(y_test, y_pred_tstr, average=avg, zero_division=0),
                recall_score(y_test, y_pred_tstr, average=avg, zero_division=0),
            ],
        }).set_index('Metric').round(4)

    else:
        y_real = y_real.astype(float)
        y_syn  = y_syn.astype(float)
        y_test = y_test.astype(float)

        baseline = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED)
        baseline.fit(X_real, y_real)
        y_pred_base = baseline.predict(X_test)

        tstr_model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED)
        tstr_model.fit(X_syn, y_syn)
        y_pred_tstr = tstr_model.predict(X_test)

        results = pd.DataFrame({
            'Metric': ['MAE', 'RMSE'],
            'Baseline (Real Train)': [
                mean_absolute_error(y_test, y_pred_base),
                np.sqrt(mean_squared_error(y_test, y_pred_base)),
            ],
            'TSTR (Synthetic Train)': [
                mean_absolute_error(y_test, y_pred_tstr),
                np.sqrt(mean_squared_error(y_test, y_pred_tstr)),
            ],
        }).set_index('Metric').round(4)

    print(f'\n🎯 ML Utility Results — {task_type} on "{TARGET_COL}"\n')
    display(results)

In [ ]:
# Visualise results
if TARGET_COL in df_clean.columns:
    ax = results.T.plot.bar(figsize=(9, 5), color=['steelblue', 'tomato'])
    ax.set_title(f'Baseline vs TSTR — {task_type}')
    ax.set_ylabel('Score')
    if is_classification:
        ax.set_ylim(0, 1.1)
    plt.xticks(rotation=0)
    plt.legend(title='Training Data')
    plt.tight_layout()
    plt.show()

    print('\n📝 Interpretation:')
    print('A small gap between TSTR and Baseline means the synthetic data')
    print('preserves most of the predictive signal — a good sign!')
    print('A large gap means important patterns were not fully captured by CTGAN.')

---
## Stage H — Privacy Risk Checks

### Two heuristic checks:

**1. Exact-Match Rate**  
How many synthetic rows are byte-for-byte identical to a real training record?
A high rate suggests memorisation.

**2. Nearest-Neighbour Distance**  
For each synthetic row, find the closest real training row in normalised feature space.
A very low minimum distance may indicate a near-copy of a real record.

> ⚠️ **Disclaimer**: These checks are heuristics only. They do **NOT** prove anonymity and do **NOT** provide formal differential-privacy (DP) guarantees. CTGAN does not implement DP-SGD or any formal privacy mechanism. Do not treat the output as safe for public release of sensitive data without a professional privacy audit.

In [ ]:
# ── 1. Exact-Match Check ──────────────────────────────────────────────────────
exact_matches = syn_df.merge(df_train, how='inner')
n_exact = len(exact_matches)
exact_rate = n_exact / max(len(syn_df), 1) * 100

print('═' * 45)
print('  PRIVACY RISK CHECKS')
print('═' * 45)
print(f'\n1. Exact-Match Check')
print(f'   Synthetic rows      : {len(syn_df):,}')
print(f'   Exact matches       : {n_exact}')
print(f'   Exact match rate    : {exact_rate:.2f}%')

if exact_rate > 5:
    print('   ⚠️  HIGH — Investigate memorisation risk.')
elif exact_rate > 1:
    print('   🟡 MODERATE — Some exact copies present.')
else:
    print('   ✅ LOW — Few or no exact duplicates.')

In [ ]:
# ── 2. Nearest-Neighbour Distance ─────────────────────────────────────────────
cols_for_nn = [c for c in (num_cols + cat_cols)
               if c in df_train.columns and c in syn_df.columns]

le = LabelEncoder()
real_enc = df_train[cols_for_nn].copy()
syn_enc  = syn_df[cols_for_nn].copy()

for c in cat_cols:
    if c in cols_for_nn:
        all_vals = pd.concat([real_enc[c], syn_enc[c]]).astype(str)
        le.fit(all_vals)
        real_enc[c] = le.transform(real_enc[c].astype(str))
        syn_enc[c]  = le.transform(syn_enc[c].astype(str))

real_arr = real_enc.values.astype(float)
syn_arr  = syn_enc.values.astype(float)

# Normalise to [0, 1] range using training data stats
col_range = real_arr.max(axis=0) - real_arr.min(axis=0)
col_range[col_range == 0] = 1
real_norm = (real_arr - real_arr.min(axis=0)) / col_range
syn_norm  = (syn_arr  - real_arr.min(axis=0)) / col_range

# Sample 500 synthetic rows for speed
rng = np.random.default_rng(RANDOM_SEED)
if len(syn_norm) > 500:
    idx = rng.choice(len(syn_norm), 500, replace=False)
    syn_norm_sample = syn_norm[idx]
else:
    syn_norm_sample = syn_norm

nn_dists = []
for row in syn_norm_sample:
    dists = np.linalg.norm(real_norm - row, axis=1)
    nn_dists.append(dists.min())

nn_dists = np.array(nn_dists)
print(f'\n2. Nearest-Neighbour Distance (normalised Euclidean)')
print(f'   Mean NN distance : {nn_dists.mean():.4f}')
print(f'   Min  NN distance : {nn_dists.min():.4f}')
print(f'   Max  NN distance : {nn_dists.max():.4f}')

if nn_dists.min() < 0.001:
    print('   ⚠️  Some synthetic rows are near-copies of real records.')
elif nn_dists.mean() > 0.05:
    print('   ✅ Synthetic rows maintain reasonable distance from real records.')
else:
    print('   🟡 Moderate proximity — review if data is sensitive.')

In [ ]:
# Plot NN distance distribution
plt.figure(figsize=(8, 4))
plt.hist(nn_dists, bins=30, color='steelblue', edgecolor='white')
plt.axvline(nn_dists.mean(), color='red', linestyle='--', label=f'Mean = {nn_dists.mean():.4f}')
plt.xlabel('Nearest-Neighbour Distance (normalised)')
plt.ylabel('Count of Synthetic Rows')
plt.title('Privacy: Nearest-Neighbour Distance Distribution')
plt.legend()
plt.tight_layout()
plt.show()

print('\n⚠️  DISCLAIMER: These checks are heuristics only.')
print('    They do NOT prove anonymity or provide formal DP guarantees.')
print('    Do not share synthetic data from sensitive sources without a formal privacy audit.')

---
## Stage I — Final Summary Report

A consolidated summary of everything we did and the key numbers.

In [ ]:
print('═' * 60)
print('  TABULAB — FINAL SUMMARY REPORT')
print('  PS 65 College Project · CTGAN Synthetic Data Generator')
print('═' * 60)

print(f"""
DATASET
  Raw shape               : {df_raw.shape}
  After preprocessing     : {df_clean.shape}
  Training rows (CTGAN)   : {len(df_train):,}
  Test rows (held-out)    : {len(df_test):,}
  Numerical columns       : {num_cols}
  Categorical columns     : {cat_cols}

CTGAN TRAINING
  Epochs                  : {EPOCHS}
  Batch size              : {BATCH_SIZE}

SYNTHETIC DATA
  Rows generated          : {len(syn_df):,}
  Output file             : {OUTPUT_PATH}
  Null values in output   : {syn_df.isna().sum().sum()}

QUALITY (SDV Report)
  Overall score           : {overall_score * 100:.1f}%
  (0–100%, higher is better; 70%+ is generally good)

PRIVACY CHECKS
  Exact-match rate        : {exact_rate:.2f}% ({n_exact} rows)
  Mean NN distance        : {nn_dists.mean():.4f}  (normalised Euclidean)
  Min  NN distance        : {nn_dists.min():.4f}
  Formal DP guarantee     : NONE — heuristic checks only
""")

if TARGET_COL in df_clean.columns:
    print('ML UTILITY (TSTR)')
    print(f'  Task                    : {task_type}')
    print(f'  Target column           : {TARGET_COL}')
    print(results.to_string())

print("""
LIMITATIONS
  1. Training time scales with epochs and dataset size.
  2. CTGAN works best with >= 500 rows and a mix of column types.
  3. Very rare category values may not be well represented.
  4. No time-series / sequential data support.
  5. Privacy: synthetic != anonymous. No DP guarantees.
  6. On Streamlit Cloud free tier: limit to epochs<=100, rows<=50k.

═══════════════════════════════════════════════════════════
  Generated by TabuLab · SDV 2.0 · CTGAN 0.12 · Streamlit
═══════════════════════════════════════════════════════════
""")